In [19]:
from glob import glob 
import geopandas as gpd
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
counties_gdf = gpd.read_file('./data/shapefiles/tl_2024_us_county/tl_2024_us_county.shp')
# select southeast states only
state_fp_codes = {
    '28': 'MS', # Make sure Mississippi is first
    '01': 'AL',
    '05': 'AR',
    # '10': 'DE',
    # '11': 'DC',
    '12': 'FL',
    '13': 'GA',
    # '21': 'KY',
    '22': 'LA',
    # '24': 'MD',
    '37': 'NC',
    # '40': 'OK',
    '45': 'SC',
    '47': 'TN',
    # '48': 'TX',
    # '51': 'VA',
    # '54': 'WV',
}
se_counties_gdf = counties_gdf[counties_gdf['STATEFP'].isin(state_fp_codes.keys())].reset_index(drop=True)
# order accprdoing to state_fp_codes keys
se_counties_gdf['STATE_CODE'] = se_counties_gdf['STATEFP'].map(state_fp_codes)
ms_counties_gdf = se_counties_gdf[se_counties_gdf['STATE_CODE'] == 'MS'].sort_values(by='COUNTYFP')
non_ms_counties_gdf = se_counties_gdf[se_counties_gdf['STATE_CODE'] != 'MS'].sort_values(by=['STATEFP', 'COUNTYFP'])
se_counties_gdf = pd.concat([ms_counties_gdf, non_ms_counties_gdf], ignore_index=True).reset_index(drop=True)
se_counties_gdf

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,STATE_CODE
0,28,061,00695754,28061,0500000US28061,Jasper,Jasper County,06,H1,G4020,279,29860,None,A,1751636189,3130850,+32.0167482,-089.1191761,"POLYGON ((-89.07939 31.81659, -89.08322 31.816...",MS
1,28,017,00695733,28017,0500000US28017,Chickasaw,Chickasaw County,06,H1,G4020,None,None,None,A,1299608408,6380767,+33.9332767,-088.9475579,"POLYGON ((-88.93123 33.80492, -88.93125 33.801...",MS
2,28,145,00695793,28145,0500000US28145,Union,Union County,06,H1,G4020,None,None,None,A,1076417817,3369003,+34.4895325,-089.0023393,"POLYGON ((-89.24609 34.49614, -89.24602 34.497...",MS
3,28,069,00695758,28069,0500000US28069,Kemper,Kemper County,06,H1,G4020,None,None,None,A,1984403151,2182350,+32.7501361,-088.6256306,"POLYGON ((-88.62123 32.57769, -88.62389 32.577...",MS
4,28,151,00695796,28151,0500000US28151,Washington,Washington County,06,H1,G4020,None,24740,None,A,1875530531,93804060,+33.2731740,-090.9444459,"POLYGON ((-90.91446 33.09742, -90.91447 33.093...",MS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
750,47,061,01639748,47061,0500000US47061,Grundy,Grundy County,06,H1,G4020,None,None,None,A,933545796,1755421,+35.3933519,-085.7103737,"POLYGON ((-85.70942 35.28925, -85.70775 35.283...",TN
751,47,099,01639764,47099,0500000US47099,Lawrence,Lawrence County,06,H1,G4020,400,29980,None,A,1598359119,2218403,+35.2204764,-087.3965460,"POLYGON ((-87.2197 35.20364, -87.21976 35.2033...",TN
752,47,059,01639747,47059,0500000US47059,Greene,Greene County,06,H1,G4020,304,24620,None,A,1611409726,5099752,+36.1794867,-082.8475235,"POLYGON ((-82.78504 35.9871, -82.78502 35.9869...",TN
753,47,157,01639790,47157,0500000US47157,Shelby,Shelby County,06,H1,G4020,368,32820,None,A,1949650933,74596358,+35.1845529,-089.8946125,"POLYGON ((-89.63788 35.20984, -89.63787 35.209...",TN


In [25]:

data_dir_root = '/home/dhester/server/dbcenter/images/naip/scenes/'
test_row = se_counties_gdf.iloc[1]

# ortho_1-1_hc_s_ca001_2022_1
with tqdm(total=len(se_counties_gdf), desc='Finding NAIP raster paths', unit='county') as pbar:
    def get_raster_path(row, pbar=pbar):
        filename = f"ortho_1-1_hc_s_{row['STATE_CODE'].lower()}{row['COUNTYFP']}_20[0-9][0-9]_1"
        glob_pattern = os.path.join(data_dir_root, '*', row['STATE_CODE'], f'{row["STATE_CODE"].lower()}_c', filename, filename + '.tif')
        files = glob(glob_pattern)
        if len(files) == 0:
            if pbar is not None:
                pbar.update(1)
            return None
        year_path = {
        int(file.split('/')[8]): file for file in files
        }
        # select most recent year
        year = max(year_path.keys())
        if pbar is not None:
            pbar.update(1)
        return year_path[year]

    se_counties_gdf['raster_path'] = se_counties_gdf.apply(get_raster_path, axis=1)



Finding NAIP raster paths: 100%|██████████| 755/755 [00:19<00:00, 37.81county/s]


In [26]:

def get_year_from_path(row):
    path = row['raster_path']
    return int(path.split('/')[8])

se_counties_gdf['year'] = se_counties_gdf.apply(get_year_from_path, axis=1)
se_counties_gdf.to_file('./data/shapefiles/se_counties_naip_raster_paths.geojson', driver='GeoJSON', index=False)

In [27]:
print(len(se_counties_gdf))

755
